In [182]:
SEED = 42

In [183]:
import torch
from torch import nn
import random
import numpy as np

# prima di tutto, dobbiamo impostare un seed per garantire la riproducibilità dei risultati. Questo è particolarmente importante quando si lavora con modelli di machine learning, poiché molte operazioni sono stocastiche.
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
set_seed(SEED)

# Poi dobbiamo settare il device su cui vogliamo eseguire il modello. Se abbiamo una GPU disponibile, useremo quella, altrimenti useremo la CPU.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
    

Using device: cpu


In [184]:
# Prepariamo il dataset
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from torch.utils.data import DataLoader, TensorDataset

iris_dataset = load_iris()
X, y = iris_dataset.data, iris_dataset.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)

steps = [
    ('scaler', StandardScaler()),
    ('mean_imputer', SimpleImputer(strategy='mean'))
]
pipe = Pipeline(steps)
X_train = pipe.fit_transform(X_train, y_train)
X_test = pipe.transform(X_test)







In [185]:
# Conversione in tensori perchè PyTorch lavora con tensori. Inoltre, dobbiamo assicurarci che i dati siano sullo stesso device del modello.
X_train_tensor = torch.tensor(X_train, dtype=torch.float32).to(device)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32).to(device)

y_train_tensor = torch.tensor(y_train, dtype=torch.long).to(device)
y_test_tensor = torch.tensor(y_test, dtype=torch.long).to(device)


In [186]:
# Creazione del DataLoader
# A cosa serve il DataLoader? Serve a gestire il batch processing dei dati, a mescolare i dati e a fornire un'interfaccia più semplice per iterare sui dati durante l'addestramento del modello.
# Il Batch Processing è una tecnica che consiste nel dividere il dataset in piccoli batch di dati, invece di utilizzare l'intero dataset per ogni aggiornamento dei pesi del modello. Questo permette di ridurre il consumo di memoria e di accelerare l'addestramento del modello.


train_loader = TensorDataset(X_train_tensor, y_train_tensor) # TensorDataset è una classe di PyTorch che permette di creare un dataset a partire da tensori. In questo caso, stiamo creando un dataset a partire dai tensori X_train_tensor e y_train_tensor.
# Si potrebbe usare solo X_train_tensor e y_train_tensor, ma TensorDataset ci permette di creare un dataset più facilmente e di gestire meglio i dati durante l'addestramento del modello.
test_loader = TensorDataset(X_test_tensor, y_test_tensor)

print(train_loader)
# Batch size: ad esempio se batch_size = 16, significa che ogni volta che il modello aggiorna i pesi, lo fa utilizzando 16 campioni del dataset. Questo permette di ridurre il consumo di memoria e di accelerare l'addestramento del modello.
train_loader = DataLoader(train_loader, batch_size=16, shuffle=True)
test_loader = DataLoader(test_loader, batch_size=16, shuffle=False)
print(train_loader)




In [ ]:
# Costruiamo la rete:
# Loss Function = CrossEntropyLoss: è una funzione di perdita comunemente usata per problemi di classificazione multi-classe. Calcola la differenza tra le probabilità previste dal modello e le etichette reali, penalizzando maggiormente le previsioni errate.
# Optimizer = Adam: è un algoritmo di ottimizzazione che combina i vantaggi di due altri algoritmi di ottimizzazione: AdaGrad e RMSProp. È particolarmente efficace per problemi di ottimizzazione non convessi e per grandi dataset.

import torch.nn.functional as F

class IrisNet(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim): # Nel nostro caso input_dim = 4 (numero di feature del dataset Iris), hidden_dim = 10 (numero di neuroni nel layer nascosto), output_dim = 3 (numero di classi del dataset Iris).
        super(IrisNet, self).__init__() # cosa fa questa riga? Chiama il costruttore della classe base nn.Module, che è necessario per inizializzare correttamente il modello. Senza questa riga, il modello non funzionerebbe correttamente.
        
        
        self.fc1 = nn.Linear(input_dim, hidden_dim) # Crea un layer lineare (fully connected) che prende in input input_dim e produce hidden_dim output. In questo caso, input_dim = 4 (numero di feature del dataset Iris), hidden_dim = 10 (numero di neuroni nel layer nascosto).
        self.dropout = nn.Dropout(p=0.5) # Aggiunge un layer di dropout con probabilità di 0.5. Il dropout è una tecnica di regolarizzazione che consiste nel "spegnere" casualmente alcuni neuroni durante l'addestramento, per evitare l'overfitting.
        self.fc2 = nn.Linear(hidden_dim, hidden_dim) # Crea un layer lineare (fully connected) che prende in input hidden_dim e produce output_dim output. In questo caso, hidden_dim = 10 (numero di neuroni nel layer nascosto), output_dim = 3 (numero di classi del dataset Iris).
        
        self.fc3 = nn.Linear(hidden_dim, output_dim) # Crea un layer lineare (fully connected) che prende in input hidden_dim e produce output_dim output. In questo caso, hidden_dim = 10 (numero di neuroni nel layer nascosto), output_dim = 3 (numero di classi del dataset Iris).
    def forward(self, x): # Definisce il passaggio in avanti del modello. In questo caso, il passaggio in avanti consiste nel passare l'input x attraverso i due layer lineari e applicare la funzione di attivazione ReLU tra i due layer.
        x = F.relu(self.fc1(x)) # Applica la funzione di attivazione ReLU al primo layer lineare. La funzione ReLU (Rectified Linear Unit) è una funzione di attivazione che restituisce 0 per input negativi e l'input stesso per input positivi. È comunemente usata nelle reti neurali perché aiuta a prevenire il problema del vanishing gradient.
        x = self.dropout(x) # Applica il layer di dropout all'output del primo layer.
        
        x = F.relu(self.fc2(x)) # Passa l'output del primo layer attraverso il secondo layer lineare.
        x = F.relu(self.fc3(x)) # Passa l'output del secondo layer attraverso il terzo layer lineare.
        return x # Restituisce l'output finale del modello.
    
model = IrisNet(input_dim=4, hidden_dim=8, output_dim=3).to(device) # Crea un'istanza del modello IrisNet e la sposta sul device (CPU o GPU) che abbiamo scelto in precedenza.


num_params = 0
for x in model.parameters():
    num_params += len(torch.flatten(x))
print(f"number of parameters: {num_params: }")


number of parameters:  73


In [188]:
from torch import optim

criterion = nn.CrossEntropyLoss() # Crea un'istanza della funzione di perdita CrossEntropyLoss. Questa funzione di perdita è comunemente usata per problemi di classificazione multi-classe.
optimizer = optim.Adam(model.parameters(), lr=0.001) # Crea un'istanza dell'ottimizzatore Adam. Questo ottimizzatore aggiorna i pesi del modello durante l'addestramento, cercando di minimizzare la funzione di perdita. Il learning rate (lr) è un iperparametro che controlla la dimensione dei passi che l'ottimizzatore fa nella direzione del gradiente.



In [ ]:
EPOCHES = 200

for epoch in range(EPOCHES):
    model.train()
    epoch_loss = 0.0
    running_accuracy = 0.0
    for batch_idx, (data, target) in enumerate(train_loader):
        # FOrward pass
        predictions = model(data)
        correct = torch.sum(target == torch.argmax(predictions, dim=1)).item() # Calcola il numero di predizioni corrette in questo batch. torch.argmax(predictions, dim=1) restituisce l'indice della classe con la probabilità più alta per ogni campione nel batch.
        running_accuracy += correct / data.size(0) # Calcola l'accuratezza di questo batch. data.size(0) restituisce il numero di campioni in questo batch.
        loss = criterion(predictions, target)
        
        # Azzera i gradienti. Questo passo è fondamentale, perché PyTorch accumula i gradienti per default. Se non azzeriamo i gradienti, i gradienti calcolati in questo batch si sommeranno a quelli calcolati nei batch precedenti, portando a un aggiornamento dei pesi errato.
        optimizer.zero_grad()
        
        # Backward pass
        loss.backward() # Calcola i gradienti della funzione di perdita rispetto ai pesi del modello. Questo passo è fondamentale per l'ottimizzazione dei pesi del modello.
        
        # Aggiorna i pesi del modello
        optimizer.step() # Aggiorna i pesi del modello utilizzando i gradienti calcolati nel passo precedente. Questo passo è fondamentale per l'ottimizzazione dei pesi del modello.
        
        epoch_loss += loss.item() # Aggiunge il valore della perdita di questo batch alla perdita totale dell'epoca. loss.item() restituisce il valore della perdita come un numero float, invece di un tensore.
        if (batch_idx) % 5 == 4:
            avg_acc_across_batches = (running_accuracy / 500)*100
            print (f'\tBatch {batch_idx + 1}, Acc {avg_acc_across_batches :.1f}%')

            running_loss = .0
            running_accuracy =.0
        
    if (epoch + 1) % 10 == 0:
            print(f"Epoch {epoch+1}/{EPOCHES} | Loss: {epoch_loss/len(train_loader):.4f}")
        
    


	Batch 5, Acc 0.3%


	Batch 5, Acc 0.3%
	Batch 5, Acc 0.3%
	Batch 5, Acc 0.4%
	Batch 5, Acc 0.3%
	Batch 5, Acc 0.3%
	Batch 5, Acc 0.3%
	Batch 5, Acc 0.3%
	Batch 5, Acc 0.4%
	Batch 5, Acc 0.4%
Epoch 10/1000 | Loss: 1.0982
	Batch 5, Acc 0.4%
	Batch 5, Acc 0.4%
	Batch 5, Acc 0.5%
	Batch 5, Acc 0.5%
	Batch 5, Acc 0.5%
	Batch 5, Acc 0.6%
	Batch 5, Acc 0.5%
	Batch 5, Acc 0.5%
	Batch 5, Acc 0.6%
	Batch 5, Acc 0.5%
Epoch 20/1000 | Loss: 1.0448
	Batch 5, Acc 0.6%
	Batch 5, Acc 0.7%
	Batch 5, Acc 0.5%
	Batch 5, Acc 0.5%
	Batch 5, Acc 0.6%
	Batch 5, Acc 0.6%
	Batch 5, Acc 0.6%
	Batch 5, Acc 0.6%
	Batch 5, Acc 0.5%
	Batch 5, Acc 0.6%
Epoch 30/1000 | Loss: 0.9858
	Batch 5, Acc 0.6%
	Batch 5, Acc 0.7%
	Batch 5, Acc 0.7%
	Batch 5, Acc 0.7%
	Batch 5, Acc 0.7%
	Batch 5, Acc 0.7%
	Batch 5, Acc 0.7%
	Batch 5, Acc 0.7%
	Batch 5, Acc 0.6%
	Batch 5, Acc 0.7%
Epoch 40/1000 | Loss: 0.9380
	Batch 5, Acc 0.8%
	Batch 5, Acc 0.7%
	Batch 5, Acc 0.7%
	Batch 5, Acc 0.8%
	Batch 5, Acc 0.7%
	Batch 5, Acc 0.7%
	Batch 5, Acc 0.7%
	Batch 5,

In [191]:
# Ora passiamo alla valutazione del modello. Per farlo dobbiamo togliere il dropout e mettere il modello in modalità di valutazione. Questo è importante perché il dropout è una tecnica di regolarizzazione che viene utilizzata solo durante l'addestramento del modello. Durante la valutazione, vogliamo che tutti i neuroni siano attivi, quindi dobbiamo disattivare il dropout.

model.eval() # Mette il modello in modalità di valutazione. Questo disattiva il dropout e altre tecniche di regolarizzazione che vengono utilizzate solo durante l'addestramento del modello.

correct = 0
total = 0

with torch.no_grad(): # Disabilita il calcolo dei gradienti. Questo è importante durante la valutazione del modello, perché non vogliamo aggiornare i pesi del modello durante la valutazione.
    for data, target in test_loader:
        predictions = model(data)
        _, predicted = torch.max(predictions.data, 1) # Restituisce l'indice della classe con la probabilità più alta. Questo è il nostro "voto" del modello per la classe dell'input.
        total += target.size(0) # Aggiunge il numero di campioni in questo batch al totale dei campioni.
        correct += (predicted == target).sum().item() # Aggiunge il numero di predizioni corrette in questo batch al totale delle predizioni corrette.

print(f"Accuracy: {100 * correct / total:.2f}%")


Accuracy: 66.67%
